<a href="https://colab.research.google.com/github/amrit-pratya/Deep_Learning_projects/blob/main/CNN_for_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# dataset & dataloader
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

# image -> scale (0, 1) -> normalize -> (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train = CIFAR10(root='./data', train=True, download=True, transform=transform)
test = CIFAR10(root='./data', train=False, download=True, transform=transform)

In [5]:
train

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False)

In [15]:
# Building the CNN

class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(128 * 4 * 4, 512),
        nn.ReLU(),

        nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1)
    x = self.fc_layers(x)

    return x


In [16]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
# Training the CNN

epochs = 10

for epoch in range(epochs):
  running_loss = 0.0
  for images, labels in train_loader:
    optimizer.zero_grad()

    outputs = model.forward(images) # FP
    loss = criterion(outputs, labels) # Loss fix
    loss.backward() # Back Propagation
    optimizer.step() # Param Updation

    running_loss += loss.item()

  print(f'epoch={epoch + 1}/{epochs} & loss: {running_loss / len(train_loader)}')

print('Finished Training')

epoch=1/10 & loss: 0.4817067758011086
epoch=2/10 & loss: 0.36258041062166013
epoch=3/10 & loss: 0.2652481733904699
epoch=4/10 & loss: 0.1901789056685041
epoch=5/10 & loss: 0.13603115948083835
epoch=6/10 & loss: 0.10883825921865604
epoch=7/10 & loss: 0.09216343920887507
epoch=8/10 & loss: 0.08264816358871281
epoch=9/10 & loss: 0.08405205305955132
epoch=10/10 & loss: 0.06640842757628435
Finished Training


In [20]:
# Evaluation

corr_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
  for images, labels in test_loader:
    outputs = model.forward(images)
    _, predicted = torch.max(outputs, 1)

    corr_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f'Accuracy of the network on the 10000 test images: {100 * corr_labels / total_labels}%')

Accuracy of the network on the 10000 test images: 75.36%
